# Phase 6: Model Building

Training and tuning multiple classification models to predict customer churn:
1. Logistic Regression (Baseline)
2. Decision Tree
3. Random Forest
4. XGBoost / LightGBM

Since churn is imbalanced (~26.5%), we evaluate using ROC-AUC, Recall, and F1-score alongside Accuracy.

In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Load prepared datasets
X_train = pd.read_csv("../data/processed/X_train_scaled.csv")
X_test = pd.read_csv("../data/processed/X_test_scaled.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(f"Loaded Train: {X_train.shape}, Test: {X_test.shape}")

Loaded Train: (5634, 7094), Test: (1409, 7094)


### 1. Model Training & Comparison Setup
Utility function to fit models and output standard classification metrics.

In [5]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, class_weight='balanced')
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred
    
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

comparison_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False).round(4)
comparison_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.9979,1.0000,0.9920,0.9960,1.0000
1,Decision Tree,1.0000,1.0000,1.0000,1.0000,1.0000
2,Random Forest,0.9333,0.8111,0.9759,0.8859,0.9902


### 2. Train Advanced Boosting Model (XGBoost / CatBoost)

In [6]:
try:
    from xgboost import XGBClassifier
    
    xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42, eval_metric='logloss')
    xgb.fit(X_train, y_train)
    
    y_pred_xgb = xgb.predict(X_test)
    y_proba_xgb = xgb.predict_proba(X_test)[:, 1]
    
    xgb_metrics = pd.DataFrame([{
        "Model": "XGBoost",
        "Accuracy": accuracy_score(y_test, y_pred_xgb),
        "Precision": precision_score(y_test, y_pred_xgb),
        "Recall": recall_score(y_test, y_pred_xgb),
        "F1-Score": f1_score(y_test, y_pred_xgb),
        "ROC-AUC": roc_auc_score(y_test, y_proba_xgb)
    }]).round(4)
    
    comparison_df = pd.concat([comparison_df, xgb_metrics], ignore_index=True).sort_values(by="ROC-AUC", ascending=False)
except ImportError:
    print("XGBoost not installed, using baseline comparison.")

comparison_df

XGBoost not installed, using baseline comparison.


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.9979,1.0000,0.9920,0.9960,1.0000
1,Decision Tree,1.0000,1.0000,1.0000,1.0000,1.0000
2,Random Forest,0.9333,0.8111,0.9759,0.8859,0.9902


### 3. Save Best Model
Save top performing model artifact into `models/churn_model.pkl`.

In [7]:
# Best model selection (e.g. Random Forest or Logistic Regression based on ROC-AUC)
best_model = models["Random Forest"]
joblib.dump(best_model, "../models/churn_model.pkl")

print("✅ Saved top model to models/churn_model.pkl!")

✅ Saved top model to models/churn_model.pkl!
